# INTEGRACION DATA SISMO - CALI - AGOSTO 2026

## 1. Aprovisionamiento

In [ ]:
%pip install -q gspread google-auth
%pip install -q rapidfuzz

## 2. Importaciones

In [ ]:
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
import random
import string

## 3. Funciones

### 3.1. Normalización de direcciones

In [ ]:
import re

def normalize_address(address) -> str:
    """
    Normalizes Colombian cadastral addresses to IGAC standard (Circular 300/01).

    Output format: [TIPO_VÍA] [NÚMERO] # [NÚMERO_GENERADORA]-[PLACA], Complement
    Example: "Calle 80 No. 45-23, barrio el peñón"  →  "CL 80 # 45-23, Barrio El Peñón"
             "Kra 10 Num 15-20, apto 301"            →  "KR 10 # 15-20, Apto 301"

    Title case is applied only to the complement (text after the first comma).
    The nomenclature part (address codes + numbers) stays in uppercase.

    Source: IGAC Instructivo para Direcciones (Circular 300/01 / Resolución MEN 166/04)
    Official abbreviations: CL, KR, AV, AC, AK, DG, TV, AU, BL, CT, CQ, CV, CC, PJ, PS, PT, TC, VT, VI
    Note: official Carrera code is KR (not CR).
    """
    if address is None:
        return ""

    value = str(address).strip()
    if not value or value in {"-", " "}:
        return value

    s = value.upper()

    # Pre-process: strip internal dots from abbreviations  →  K.R.A. → KRA, C.L. → CL
    s = re.sub(
        r'\b([A-Z])\.([A-Z])(?:\.([A-Z]))?\.?',
        lambda m: m.group(1) + m.group(2) + (m.group(3) or ""),
        s,
    )

    # ── Road type → IGAC standard code ───────────────────────────────────────
    # Order matters: compound types before their components;
    # full words before short abbreviations.
    _ROAD_TYPES = [
        # Compound avenidas (must precede plain AV, CL, KR)
        (r'\bAVENIDA CALLE\b|\bAV CALLE\b|\bAV CL\b',                          'AC'),
        (r'\bAVENIDA K?ARRERA\b|\bAV K?ARRERA\b|\bAV K?R\b|\bAV CRA\b',        'AK'),
        # Autopista (AU)
        (r'\bAUTOPISTA\b|\bAUTOP\b|\bAUT\b',                                   'AU'),
        # Avenida (AV)
        (r'\bAVENIDA\b|\bAVDA\b|\bAVD\b|\bAVE\b|\bAV\b',                       'AV'),
        # Carretera (CT) — before Carrera to avoid CARR collision
        (r'\bCARRETERA\b|\bCARRET\b',                                           'CT'),
        # Carrera (KR) — official IGAC code is KR, not CR
        (r'\bCARRERA\b|\bKARRERA\b|\bCARR\b|\bCRA\b|\bKRA\b|\bKRR\b|\bKR\b|\bCR\b', 'KR'),
        # Calle (CL)
        (r'\bCALLE\b|\bCALL\b|\bCLLE\b|\bCLL\b|\bCL\b',                        'CL'),
        # Circunvalar (CV) — before Circular to avoid CQ collision
        (r'\bCIRCUNVALAR\b|\bCIRCUNV\b|\bCIRCV\b',                            'CV'),
        # Circular (CQ)
        (r'\bCIRCULAR\b|\bCIRC\b',                                             'CQ'),
        # Diagonal (DG)
        (r'\bDIAGONAL\b|\bDIAG\b|\bDG\b',                                      'DG'),
        # Transversal (TV)
        (r'\bTRANSVERSAL\b|\bTRANSV\b|\bTRANS\b|\bTRAV\b|\bTV\b|\bTR\b',      'TV'),
        # Troncal (TC)
        (r'\bTRONCAL\b|\bTRONC\b',                                              'TC'),
        # Bulevar (BL)
        (r'\bBULEVAR\b|\bBOULEVAR\b|\bBLVD\b|\bBL\b',                        'BL'),
        # Pasaje (PJ) — before Paseo
        (r'\bPASAJE\b|\bPSJE\b|\bPJE\b|\bPJ\b',                               'PJ'),
        # Paseo (PS)
        (r'\bPASEO\b|\bPSO\b',                                                  'PS'),
        # Peatonal (PT)
        (r'\bPEATONAL\b|\bPEAT\b',                                             'PT'),
        # Variante (VT)
        (r'\bVARIANTE\b',                                                        'VT'),
        # Vía (VI)
        (r'\bV[IÍ]A\b',                                                          'VI'),
        # Cuentas Corridas (CC)
        (r'\bCUENTAS? CORRIDAS?\b',                                             'CC'),
    ]
    for pattern, code in _ROAD_TYPES:
        s = re.sub(pattern, code, s)

    # Strip stray trailing dots left after code substitution  →  "CL." → "CL"
    s = re.sub(
        r'\b(AU|AV|AC|AK|KR|CL|CV|CQ|CT|DG|TV|TC|BL|PJ|PS|PT|VT|VI|CC)\.',
        r'\1',
        s,
    )

    # ── Número sign (# separator) ─────────────────────────────────────────────
    # Word-bounded variants: NUMERO, NUM, NRO, NO, NN
    # (?![A-ZÁÉÍÓÚ]) guards against eating the start of a longer word:
    # without it, "NORTE-2" → "# RTE-2" because \bNO matches inside NORTE
    s = re.sub(
        r'\b(?:N[ÚU]MERO|NUMERO|N[ÚU]M|NUM|NRO|NO|NN)(?![A-ZÁÉÍÓÚ])\.?\s*',
        '# ',
        s,
        flags=re.UNICODE,
    )
    # Non-word variants: N°, Nº  (no \b needed around the degree/ordinal sign)
    s = re.sub(r'N[°º]\.?\s*', '# ', s)

    # ── Whitespace normalization ──────────────────────────────────────────────
    s = re.sub(r'\s*#\s*', ' # ', s)   # exactly one space on each side of #
    s = re.sub(r'\s+', ' ', s).strip()

    # ── Casing: nomenclature stays uppercase; complement (after comma) → title ─
    if ',' in s:
        nomenclatura, complement = s.split(',', 1)
        return nomenclatura.strip() + ', ' + complement.strip().title()
    return s

### 3.2. Handshake y Vectorización

In [ ]:
import numpy as np

# ── Handshake (canonical fingerprint string) ──────────────────────────────────
_HANDSHAKE_RE = re.compile(
    r'^(AU|AV|AC|AK|KR|CL|CV|CQ|CT|DG|TV|TC|BL|PJ|PS|PT|VT|VI|CC)'
    r'\s+(\d+[A-Z]?(?:\s?BIS)?)'
    r'(?:\s+#\s+(\d+[A-Z]?)(?:[-\s](\d+))?)?',
    re.IGNORECASE,
)

def make_handshake(address) -> str:
    """
    Extracts a compact canonical fingerprint from a normalized Colombian address.
    Strips complement (text after comma) — only the nomenclature part is used.

    Examples:
      "CL 5 # 43A-15, El Lido, Cali"          →  "CL5#43A-15"
      "KR 57 # 3-88, Cuarto De Legua, Cali"   →  "KR57#3-88"
      "CL 10 BIS # 70 34, El Limonar, Cali"   →  "CL10BIS#70-34"
      "CL 3 # 61B, Pampalinda, Cali"           →  "CL3#61B"
      "KR 72 CON CL 11, La Hacienda, Cali"     →  "KR72"
    """
    if not address or str(address).strip() in {"", "-"}:
        return ""
    nomenclatura = str(address).split(",")[0].strip().upper()
    m = _HANDSHAKE_RE.match(nomenclatura)
    if m:
        road_type = m.group(1)
        road_num  = m.group(2).replace(" ", "")
        cross_num = m.group(3) or ""
        distance  = m.group(4) or ""
        if cross_num and distance:
            return f"{road_type}{road_num}#{cross_num}-{distance}"
        if cross_num:
            return f"{road_type}{road_num}#{cross_num}"
        return f"{road_type}{road_num}"
    return re.sub(r'\s+', '', nomenclatura)

# ── Vector representation (3D) — three-pass parser ───────────────────────────

# Pass 1: road type + number + optional suffix (letter OR BIS, with optional space)
_ROAD_HEAD_RE = re.compile(
    r'^(AU|AV|AC|AK|KR|CL|CV|CQ|CT|DG|TV|TC|BL|PJ|PS|PT|VT|VI|CC)\s+'
    r'(\d+)\s*(BIS\b|[A-Z](?![A-Z]))?',
    re.IGNORECASE,
)

# Pass 2a: first valid "# NUMBER [LETTER] [-+ DISTANCE]" anywhere in the remainder
# skips qualifier tokens (NTE., RTE, SUR, CON, etc.) that appear right after #
# [-\s]+ handles multi-char separators like " - " (space-dash-space)
_CROSS_RE = re.compile(
    r'#\s*(?:[A-Z][A-Z0-9]*\.?\s+)*(\d+)\s*([A-Z](?![A-Z]))?(?:[-\s]+(\d+))?',
    re.IGNORECASE,
)

# Pass 2b: if cross found but distance still missing, extract from "# QUALIFIER-NUMBER"
# e.g. "# RTE-2" → distance=2
_DIST_QUALIFIER_RE = re.compile(
    r'#\s*[A-Z][A-Z0-9]*\.?-(\d+)',
    re.IGNORECASE,
)

# Pass 2c: no # at all — parse implicit "CROSS [LETTER] [DIGIT] DISTANCE" sequence
# absorbs embedded digit in cross (e.g. 1A9 → cross=1A, noise=9, distance=next token)
_CROSS_NO_HASH_RE = re.compile(
    r'(?<!\w)(\d{1,3})\s*([A-Z](?![A-Z]))?\d*[-\s]+(\d{1,3})(?!\d)',
    re.IGNORECASE,
)

_ROAD_CODES = {
    'CL':1,'KR':2,'AV':3,'AC':4,'AK':5,'DG':6,'TV':7,'AU':8,'BL':9,
    'CT':10,'CQ':11,'CV':12,'CC':13,'PJ':14,'PS':15,'PT':16,'TC':17,'VT':18,'VI':19,
}

def _to_decimal(num_str, letter=None, bis=False) -> float:
    """
    5    → 5.0
    5A   → 5.01   (A=+0.01, B=+0.02, ..., Z=+0.26)
    5 BIS→ 5.50   (BIS = mid-point between N and N+1)
    """
    if not num_str:
        return 0.0
    base = float(num_str)
    if bis:
        return base + 0.50
    if letter:
        return base + (ord(letter.upper()) - ord('A') + 1) * 0.01
    return base

def address_to_vector(address) -> np.ndarray:
    """
    Converts a normalized Colombian address to a 3D vector.

    Dimensions:
      [0] road_full  = road_type_code * 1000 + road_num_decimal
                       CL 5A → 1005.01  |  KR 60A → 2060.01
      [1] cross_full = cross_num_decimal
                       43A → 43.01  |  61B → 61.02
      [2] distance   = placa distance from corner

    Euclidean distance semantics:
      Different road type   → Δ ≥ 1000   (completely different road)
      Adjacent road number  → Δ = 1      (one block)
      Letter suffix         → Δ = 0.01   (sub-block subdivision)
      BIS suffix            → Δ = 0.50   (mid-block duplicate)

    Three-pass parsing strategy:
      1. Road head from start (type + num + letter/BIS, space-tolerant)
      2a. Search for "# NUMBER" — skips qualifier tokens (NTE, RTE, CON, etc.)
          [-\s]+ handles separators like " - " (space-dash-space)
      2b. If distance still missing, extract from "# QUALIFIER-NUMBER" (e.g. # RTE-2)
      2c. No # found — fallback to implicit CROSS DISTANCE sequence (e.g. 1A9 80)
    """
    null_vec = np.zeros(3, dtype=np.float32)
    if not address or str(address).strip() in {"", "-"}:
        return null_vec
    nomenclatura = str(address).split(",")[0].strip().upper()

    head = _ROAD_HEAD_RE.match(nomenclatura)
    if not head:
        return null_vec

    road_type, road_num, suffix = head.groups()
    bis         = bool(suffix) and suffix.upper() == 'BIS'
    road_letter = suffix if (suffix and not bis) else None
    road_code   = _ROAD_CODES.get(road_type.upper(), 0)

    rest = nomenclatura[head.end():]
    cross_num = cross_letter = distance = None

    cross_m = _CROSS_RE.search(rest)
    if cross_m:
        cross_num, cross_letter, distance = cross_m.groups()
        if not distance:
            dist_m = _DIST_QUALIFIER_RE.search(rest)
            if dist_m:
                distance = dist_m.group(1)
    else:
        no_hash_m = _CROSS_NO_HASH_RE.search(rest)
        if no_hash_m:
            cross_num, cross_letter, distance = no_hash_m.groups()

    return np.array([
        road_code * 1000 + _to_decimal(road_num, road_letter, bis),
        _to_decimal(cross_num, cross_letter),
        float(distance) if distance else 0.0,
    ], dtype=np.float32)

def add_handshake(df, source_col="direccion_norm"):
    """Inserts integration_handshake (3D vector as tuple) immediately after source_col."""
    loc = df.columns.get_loc(source_col) + 1
    df.insert(loc, "integration_handshake",
              df[source_col].apply(lambda x: tuple(address_to_vector(x))))
    return df

### 3.3. Parseo de coordenadas a WGS84

In [ ]:
import re
from typing import Optional, Tuple

# ── Empty / non-parseable sentinel values ────────────────────────────────────
_COORD_EMPTY = frozenset({
    '', '-', ' ', 'n/a', 'na', 'nd', 's/d',
    'sin dato', 'ninguno', 'no tengo', 'sin coordenadas', 'no aplica',
})

# ── Unicode quote normalization table ────────────────────────────────────────
# Run FIRST in parse_coords. After this, only ASCII ' and " are in the string,
# so the DMS regex uses plain ' and " without any escaping gymnastics.
_QUOTE_NORM = str.maketrans({
    '‘': "'", '’': "'", 'ʼ': "'", '′': "'",  # smart/prime → '
    '“': '"', '”': '"', 'ʺ': '"', '″': '"',  # smart/double-prime → "
})

# ── Google Maps URL patterns ─────────────────────────────────────────────────
_GMAPS_AT = re.compile(r'@(-?\d{1,3}\.\d+),(-?\d{1,3}\.\d+)')
_GMAPS_Q  = re.compile(r'[?&]q=(-?\d{1,3}\.\d+)[,+](-?\d{1,3}\.\d+)', re.IGNORECASE)
_GMAPS_LL = re.compile(r'[?&]ll=(-?\d{1,3}\.\d+),(-?\d{1,3}\.\d+)', re.IGNORECASE)

# ── Coordinate label stripper ─────────────────────────────────────────────────
# Handles: LATITUD, LATITUD:, LATITUD :, LAT:, LON:, LONGITUD:, etc.
_COORD_LABELS = re.compile(
    r'\b(?:LATITUD|LONGITUD|LAT|LON|LONG)\b\s*:?\s*', re.IGNORECASE
)

# ── DMS regex ─────────────────────────────────────────────────────────────────
# Built with adjacent string literals so ' and " appear as plain ASCII chars,
# avoiding the double-quote-inside-double-quoted-raw-string escaping trap.
# Input must already be normalized (Unicode quotes → ASCII via _normalize).
#
# Matches:  3°27'24.5"N 76°32'35.2"W
#           3°22'22.02"N 76°33'13"O        (O = Oeste = West)
#           3°22'20.67"N 76°33'17.41"W     (after label-stripping)
_DMS = re.compile(
    r'(\d{1,3})\s*[°*]\s*(\d{1,2})\s*'
    "'"
    r'\s*(\d{1,2}(?:[.,]\d+)?)\s*'
    '"'
    r'{0,2}\s*([NSns])'
    r'[\s,;]+'
    r'(\d{1,3})\s*[°*]\s*(\d{1,2})\s*'
    "'"
    r'\s*(\d{1,2}(?:[.,]\d+)?)\s*'
    '"'
    r'{0,2}\s*([EWOewo])',
)

# ── Decimal degrees (dot separator) ─────────────────────────────────────────
_DD = re.compile(r'(-?\d{1,3}\.\d+)\s*[,;\s]\s*(-?\d{1,3}\.\d+)')

# ── MAGNA-SIRGAS planar (EPSG:3115) ──────────────────────────────────────────
_PLANAR_LABELED = re.compile(
    r'(?:[EN][:\s]*)(\d{6,7}(?:[.,]\d+)?)\s*[,;\s]\s*(?:[EN][:\s]*)(\d{6,7}(?:[.,]\d+)?)',
    re.IGNORECASE,
)
_PLANAR_BARE = re.compile(
    r'(?<!\d)(\d{6,7}(?:\.\d+)?)\s*[,;\s]\s*(\d{6,7}(?:\.\d+)?)(?!\d)'
)


def _normalize(s: str) -> str:
    """Normalize Unicode quotes to ASCII and collapse all whitespace to single space."""
    return re.sub(r'\s+', ' ', s.translate(_QUOTE_NORM)).strip()


def _fix_decimal(s: str) -> float:
    return float(str(s).replace(',', '.'))


def _dms_to_dd(deg: str, mins: str, secs: str, hemi: str) -> float:
    dd = float(deg) + float(mins) / 60.0 + _fix_decimal(secs) / 3600.0
    return -dd if hemi.upper() in ('S', 'W', 'O') else dd


def _valid_wgs84(lat: float, lon: float) -> bool:
    return -90.0 <= lat <= 90.0 and -180.0 <= lon <= 180.0


def _plausible_cali(lat: float, lon: float) -> bool:
    """Bounding box for Valle del Cauca and surroundings."""
    return 2.0 <= lat <= 5.5 and -78.0 <= lon <= -75.0


def _try_gmaps_url(s: str) -> Optional[Tuple[float, float]]:
    for pat in (_GMAPS_AT, _GMAPS_Q, _GMAPS_LL):
        m = pat.search(s)
        if m:
            lat, lon = float(m.group(1)), float(m.group(2))
            if _valid_wgs84(lat, lon):
                return lat, lon
    return None


def _try_dms(s: str) -> Optional[Tuple[float, float]]:
    # Strip LATITUD/LONGITUD labels, then collapse remaining whitespace
    clean = re.sub(r'\s+', ' ', _COORD_LABELS.sub(' ', s)).strip()
    m = _DMS.search(clean)
    if not m:
        return None
    lat = _dms_to_dd(m.group(1), m.group(2), m.group(3), m.group(4))
    lon = _dms_to_dd(m.group(5), m.group(6), m.group(7), m.group(8))
    return (lat, lon) if _valid_wgs84(lat, lon) else None


def _try_decimal(s: str) -> Optional[Tuple[float, float]]:
    # First pass: standard dot-decimal  "3.491, -76.519"
    m = _DD.search(s)
    if not m:
        # Second pass: comma-as-decimal-separator  "3,4910915, -76,5191746"
        # Replace digit,digit → digit.digit (field separator ", " is unaffected
        # because it has a space after the comma)
        m = _DD.search(re.sub(r'(\d),(\d)', r'\1.\2', s))
    if not m:
        return None
    a, b = float(m.group(1)), float(m.group(2))
    for lat, lon in ((a, b), (b, a)):
        if _valid_wgs84(lat, lon) and _plausible_cali(lat, lon):
            return lat, lon
    for lat, lon in ((a, b), (b, a)):
        if _valid_wgs84(lat, lon):
            return lat, lon
    return None


def _try_magna_sirgas(s: str) -> Optional[Tuple[float, float]]:
    """MAGNA-SIRGAS / Colombia West (EPSG:3115) → WGS84. No-op if pyproj absent."""
    try:
        from pyproj import Transformer
    except ImportError:
        return None
    m = _PLANAR_LABELED.search(s) or _PLANAR_BARE.search(s)
    if not m:
        return None
    e, n = _fix_decimal(m.group(1)), _fix_decimal(m.group(2))
    in_range_en = 900_000 <= e <= 1_300_000 and 700_000 <= n <= 1_200_000
    in_range_ne = 900_000 <= n <= 1_300_000 and 700_000 <= e <= 1_200_000
    if not (in_range_en or in_range_ne):
        return None
    if in_range_ne and not in_range_en:
        e, n = n, e
    try:
        t = Transformer.from_crs('EPSG:3115', 'EPSG:4326', always_xy=True)
        lon, lat = t.transform(e, n)
        return (lat, lon) if _valid_wgs84(lat, lon) else None
    except Exception:
        return None


def parse_coords(value) -> Optional[Tuple[float, float]]:
    """
    Parses a coordinate string → (lat, lon) WGS84 decimal degrees, or None.

    Pipeline (all formats handled after Unicode normalization):
      1. Google Maps URL   →  /@lat,lon  |  ?q=lat,lon  |  &ll=lat,lon
      2. DMS               →  3°27'24.5"N 76°32'35.2"W  |  3°22'13"O
         DMS labeled       →  LATITUD 3°22'20.67"N LONGITUD 76°33'17.41"W
         DMS label+colon   →  LATITUD: 3°22'20.73"N LONGITUD: 76°33'17.80"W
      3. Decimal dot       →  3.456789, -76.543210
         Decimal comma     →  3,4910915, -76,5191746
      4. MAGNA-SIRGAS / Colombia West (EPSG:3115)  —  requires pyproj
    """
    if value is None:
        return None
    raw = str(value).strip()
    if not raw:
        return None
    s = _normalize(raw)
    if not s or s.lower() in _COORD_EMPTY:
        return None
    return (
        _try_gmaps_url(s)
        or _try_dms(s)
        or _try_decimal(s)
        or _try_magna_sirgas(s)
    )


def coords_to_wgs84(value) -> str:
    """Returns 'lat, lon' in WGS84 decimal, or '' if the value can't be parsed."""
    result = parse_coords(value)
    if result is None:
        return ''
    lat, lon = result
    return f'{lat:.6f}, {lon:.6f}'


## 4. Procesamiento de datos "EDAN"

### 4.1. Lectura

In [ ]:
SPREADSHEET_ID = "1QRLezOtMTZpePluDl7VVzxINYLgvACVB76z54vJNpfE"
SHEET_NAME = "EDAN 100826 - Datos Madre"
SERVICE_ACCOUNT_FILE = "service_account.json"

creds = Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"],
)
gc = gspread.authorize(creds)

sheet = gc.open_by_key(SPREADSHEET_ID).worksheet(SHEET_NAME)
data = sheet.get_all_records()
df = pd.DataFrame(data)

print(f"Shape: {df.shape}")
#df.head()

### 4.2. Limpieza de datos

In [ ]:
df = df.drop(columns=["DESCRIPCION CENTRALIZADA", 
                      "REFERENCIADO EN EL MAPA", 
                      "ID",
                      "Direccion",
                      "Color semaforo"
                      ])

In [ ]:
df = df.rename(columns={
    "Prioridad": "prioridad",
    "Estado": "estado",
    "Comuna - Corregimiento": "comuna_corregimiento",
    "Zona": "zona",
    "Barrio": "barrio_vereda",
    "Tipo": "tipo_estructura",
    "Referencia": "punto_referencia",
    "DIRECCION COMPLETA": "direccion",
    "Descripción": "descripcion",
    "Normalizacion": "normalizacion",
    "Personas Evacuadas": "n_personas_evacuadas",
    "Seres sintientes": "n_seres_sintientes",
    "Atencion de la respuesta": "personal_atencion",
    "Colapso Total": "n_colapsados_total",
    "Colapso Parcial": "n_colapsados_parcial",
    "Daños": "n_danos",
    "Atrapamientos": "n_atrapamientos",
    "Rescatados": "n_rescatados",
    "Fallecidos": "n_fallecidos",
    "NOMBRE DELEGADO": "nombre_delegado",
    "OBSERVACIONES (Gestiòn del riesgo)": "observaciones_gred",
    "DESAPARECIDOS": "n_desaparecidos",
    "COORDENADAS": "coords",
    "DESCRIPCION DE LA SITUACION": "descripcion_2",
    "Observaciones generales (FORMULARIO)": "observaciones_generales",
    "NOMBRE LIDER": "nombre_lider",
    "TELEFONO": "num_telefono",
    "GRUPO ": "grupo",
})

In [ ]:
df["prioridad"] = df["prioridad"].str.title()
df["direccion"] = df["direccion"].str.title()
df["nombre_lider"] = df["nombre_lider"].str.title()
df["nombre_delegado"] = df["nombre_delegado"].str.title()

In [ ]:
empty_values = {"", "-", " "}

cols_check = [c for c in df.columns if c != "sitio_id"]

anulados = df["estado"].str.upper() == "ANULADO"
sin_datos = df[cols_check].apply(
    lambda row: all(str(v).strip() in empty_values for v in row), axis=1
)

print(f"Registros anulados:  {anulados.sum()}")
print(f"Registros sin datos: {sin_datos.sum()}")
print(f"Total a eliminar:    {(anulados | sin_datos).sum()}")

df = df[~(anulados | sin_datos)].reset_index(drop=True)
print(f"Registros restantes: {len(df)}")

### 4.3. Asignación de índices artificiales y orden

In [ ]:
rng = random.Random(42)
chars = string.ascii_uppercase + string.digits

seen = set()
sitio_ids = []
while len(sitio_ids) < len(df):
    candidate = "".join(rng.choices(chars, k=4))
    if candidate not in seen:
        seen.add(candidate)
        sitio_ids.append(candidate)

df.insert(0, "sitio_id", sitio_ids)
#df.head(10)

In [ ]:
df.shape

In [ ]:
def group_after(cols, anchor, companion):
    cols = [c for c in cols if c != companion]
    cols.insert(cols.index(anchor) + 1, companion)
    return cols

all_cols = df.columns.tolist()
n_cols = [c for c in all_cols if c.startswith("n_")]
anchor = all_cols.index("n_personas_evacuadas")
before = [c for c in all_cols[:anchor] if not c.startswith("n_")]
after = [c for c in all_cols[anchor:] if not c.startswith("n_")]

order = before + n_cols + after
order = group_after(order, "descripcion", "descripcion_2")
df = df[order]

### 4.4. Normalización de direcciones y Vectorización

In [ ]:
df.insert(df.columns.get_loc("direccion") + 1, "direccion_norm", df["direccion"].apply(normalize_address))

In [ ]:
#df['direccion_norm'].head()

In [ ]:
df = add_handshake(df, source_col="direccion_norm")
df[["direccion_norm", "integration_handshake"]].head(10)

In [ ]:
df["coords"] = df["coords"].apply(lambda v: coords_to_wgs84(v) or v)

In [ ]:
df_edan = df

## 5. Procesamiento de datos "Visitas"

### 5.1. Lectura

In [ ]:
SPREADSHEET_ID = "1SIzarDbjtaD6JVM7cUHWqrcLBJNgYj6ZooHVU9tRKN4"
SHEET_NAME = "Respuestas de formulario 1"
SERVICE_ACCOUNT_FILE = "service_account.json"

creds = Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"],
)
gc = gspread.authorize(creds)

sheet = gc.open_by_key(SPREADSHEET_ID).worksheet(SHEET_NAME)
data = sheet.get_all_records()
df = pd.DataFrame(data)

print(f"Shape: {df.shape}")

### 5.2. Limpieza de datos

In [ ]:
df = df.drop(columns=["Marque con una X si este formulario fue diligenciado por el grupo de arquitectos e ingenieros voluntarios para la evaluación de edificaciones en riesgo de colapso.\n\nDE LO CONTRATIO DEJA SIN MARCAR LA CASILLA",
                      "Columna 22", 
                      "Columna 1",
                      "Colapso"
                      ])

In [ ]:
df = df.rename(columns={
    "Marca temporal": "timestamp",
    "Dirección de correo electrónico": "email",
    "Dirección completa": "direccion",
    "Coordenadas Geográficas": "coords",
    "Descripción completa de la situación encontrada": "descripcion",
    "Cantidad de personas fallecidos (EN NÚMEROS POR FAVOR)": "n_fallecidos",
    "Cantidad de personas atrapadas  (EN NÚMEROS POR FAVOR)": "n_atrapamientos",
    "Cantidad de personas rescatadas  (EN NÚMEROS POR FAVOR)": "n_rescatados",
    "Suba los soportes que considere necesarios (Vídeos, fotos, audios, documentos)": "evidencia_soporte",
    "Nombre de la persona que diligencia el reporte": "nombre_diligenciador",
    "¿Se necesita evacuación preventiva de las personas?": "evacuacion_preventiva",
    "Cantidad de personas aproximadas que  necesitan evacuar (POR FAVOR DIGITALICE EN NÚMEROS) \n\nSi al momento de la visita, la edificación ya fué evacuada, diligenciar en observaciones y colocar el aproximado de personas que evacuaron (si cuenta con el dato)": "personas_necesitan_evacuar",
    "Cantidad de Casas, vivienda o apartamentos caracterizados y atendidos  (POR FAVOR DIGITALICE EN NÚMEROS) ": "n_estructuras_caracterizadas",
    "Nombre del Edificio, Centro Comercial, Hospital, Unidad o Conjunto Residencial (SI APLICA, en caso contrario diligenciar como NA)": "nombre_estructura",
    "Nombre del organismo que pertenece. \n\nEn caso de ser parte del grupo de ingenieros y arquitectos voluntarios ir a la siguiente pregunta": "nombre_organismo",
    "Nivel de riesgo ": "nivel_riesgo",
    "Consecutivo - id (solo si fue asignado)": "consecutivo_gred",
    "¿La edificación requiere demolición?": "requiere_demolicion",
    "Describa las observaciones que considere pertinentes a la pregunta anterior": "observaciones",
    "Requieren evacuación": "requieren_evacuacion",
    "Observaciones generales":"observaciones_generales",
    "Comuna": "comuna_corregimiento",
    "Barrio": "barrio_vereda",
    "Tipo de edificación atendida": "tipo_estructura",
    "Situación de la vivienda": "estado_estructura"
    
})

In [ ]:
df.columns

In [ ]:
df["barrio_vereda"] = df["barrio_vereda"].str.title()
df["comuna_corregimiento"] = df["comuna_corregimiento"].str.title()
df["tipo_estructura"] = df["tipo_estructura"].str.title()
df["nombre_diligenciador"] = df["nombre_diligenciador"].str.title()
df["nombre_estructura"] = df["nombre_estructura"].str.title()
df["nombre_diligenciador"] = df["nombre_diligenciador"].str.title()
df["estado_estructura"] = df["estado_estructura"].str.title()
df["requiere_demolicion"] = df["requiere_demolicion"].str.title()
df["direccion"] = df["direccion"].str.title()

In [ ]:
df.head()

### 5.3. Asignación de índices artificiales y orden

In [ ]:
rng = random.Random(88)
chars = string.ascii_uppercase + string.digits

seen = set()
visita_ids = []
while len(visita_ids) < len(df):
    candidate = "".join(rng.choices(chars, k=4))
    if candidate not in seen:
        seen.add(candidate)
        visita_ids.append(candidate)

df.insert(0, "visita_id", visita_ids)
df.head()

### 5.4. Normalización de direcciones y Vectorización

In [ ]:
df.insert(df.columns.get_loc("direccion") + 1, "direccion_norm", df["direccion"].apply(normalize_address))

In [ ]:
_empty = {"", "-", " "}

def _with_location(row):
    norm = str(row["direccion_norm"]).strip()
    if norm in _empty:
        return norm
    barrio = str(row["barrio_vereda"]).strip()
    if barrio and barrio not in _empty:
        return f"{norm}, {barrio}, Cali"
    return f"{norm}, Cali"

df["direccion_norm"] = df.apply(_with_location, axis=1)

In [ ]:
df = add_handshake(df, source_col="direccion_norm")
df[["direccion_norm", "integration_handshake"]].head(10)

In [ ]:
df["coords"] = df["coords"].apply(lambda v: coords_to_wgs84(v) or v)

In [ ]:
df_visitas = df

## 6. Integración

### 6.1. Método de matching

Un join exacto por `direccion_norm` falla entre fuentes con digitación tan heterogénea. El emparejamiento se hace en 3 niveles — el primero que acierta gana:

1. **`handshake`** — igualdad exacta de la huella canónica (`make_handshake`): mismo tipo de vía, número, cruce y placa. Solo se usan huellas *fuertes* (que contienen `#`); una huella como `KR72` sin cruce emparejaría cualquier predio de esa carrera.
2. **`vector`** — distancia euclidiana entre los vectores 3D de `integration_handshake` ≤ `vector_tol`. Tolera diferencias a nivel de letra/BIS (sub-cuadra). Igual que arriba, requiere cruce (`vector[1] != 0`) en ambos lados.
3. **`fuzzy`** — similitud de texto (`rapidfuzz` `token_sort_ratio`) sobre `direccion_norm` completa (incluye barrio) ≥ `fuzzy_threshold`. Es el fallback para direcciones que el parser no pudo vectorizar (vector nulo o sin cruce).

Cada visita se empareja con **máximo un** sitio EDAN. Varias visitas pueden apuntar al mismo sitio (visitas repetidas al mismo predio), lo cual es correcto operativamente.

In [ ]:
from rapidfuzz import process, fuzz

def build_match_table(df_edan, df_visitas, vector_tol=0.05, fuzzy_threshold=92.0):
    """
    Matches each visita to at most one EDAN site using direccion_norm.

    Tiers (first hit wins):
      1. handshake — exact canonical fingerprint equality (make_handshake).
                     Only strong fingerprints (containing '#') are used: a key
                     like "KR72" would match any site on that road.
      2. vector    — Euclidean distance between integration_handshake vectors
                     <= vector_tol. Requires cross info (vector[1] != 0) on
                     both sides for the same reason.
      3. fuzzy     — token_sort_ratio on full direccion_norm (includes barrio)
                     >= fuzzy_threshold. Fallback for weak/unparseable addresses.

    match_score semantics per tier:
      handshake → 100.0 | vector → distance (lower is better) | fuzzy → ratio 0-100

    Returns one row per visita: visita_id | sitio_id | match_method | match_score
    """
    # Tier 1 index: strong handshake -> first sitio_id
    hs_index = {}
    for sid, key in zip(df_edan["sitio_id"], df_edan["direccion_norm"].apply(make_handshake)):
        if key and "#" in key and key not in hs_index:
            hs_index[key] = sid

    # Tier 2 index: vectors with cross info
    vecs = np.array([list(v) for v in df_edan["integration_handshake"]], dtype=np.float64)
    has_cross = vecs[:, 1] != 0.0
    edan_vecs = vecs[has_cross]
    edan_vec_sids = df_edan["sitio_id"].to_numpy()[has_cross]

    # Tier 3 index: non-empty normalized addresses
    has_text = df_edan["direccion_norm"].astype(str).str.strip().ne("")
    edan_texts = df_edan.loc[has_text, "direccion_norm"].tolist()
    edan_text_sids = df_edan.loc[has_text, "sitio_id"].tolist()

    rows = []
    for _, visita in df_visitas.iterrows():
        addr = str(visita["direccion_norm"]).strip()
        sid = method = score = None

        key = make_handshake(addr) if addr else ""
        if key and "#" in key and key in hs_index:
            sid, method, score = hs_index[key], "handshake", 100.0

        if sid is None:
            v = np.asarray(visita["integration_handshake"], dtype=np.float64)
            if v[1] != 0.0 and len(edan_vecs):
                dists = np.linalg.norm(edan_vecs - v, axis=1)
                best = int(np.argmin(dists))
                if dists[best] <= vector_tol:
                    sid, method = edan_vec_sids[best], "vector"
                    score = round(float(dists[best]), 4)

        if sid is None and addr and edan_texts:
            hit = process.extractOne(addr, edan_texts,
                                     scorer=fuzz.token_sort_ratio,
                                     score_cutoff=fuzzy_threshold)
            if hit:
                sid, method, score = edan_text_sids[hit[2]], "fuzzy", round(hit[1], 1)

        rows.append({"visita_id": visita["visita_id"], "sitio_id": sid,
                     "match_method": method, "match_score": score})

    return pd.DataFrame(rows)

In [ ]:
match_table = build_match_table(df_edan, df_visitas)
match_table["match_method"].value_counts(dropna=False)

### 6.2. Mezcla horizontal e índice combinado

Merge externo (`outer`) para no perder registros de ninguna fuente de verdad:

- **Sitio EDAN con visita** → fila con ambas fuentes; las columnas repetidas quedan con sufijo `_edan` / `_visita`.
- **Sitio EDAN sin visita** → columnas de visita vacías.
- **Visita sin sitio EDAN** → columnas de EDAN vacías.

El índice `registro_id` combina `sitio_id`-`visita_id`; el lado faltante se marca con `----` (imposible que colisione con un id real, que es alfanumérico).

In [ ]:
df_master = (
    df_edan
    .merge(match_table, on="sitio_id", how="outer")
    .merge(df_visitas, on="visita_id", how="outer", suffixes=("_edan", "_visita"))
)

# Combined index: sitio_id-visita_id, "----" marks the missing side
registro_id = df_master["sitio_id"].fillna("----") + "-" + df_master["visita_id"].fillna("----")
df_master.insert(0, "registro_id", registro_id)
df_master = df_master.set_index("registro_id")

print(f"EDAN: {len(df_edan)} | Visitas: {len(df_visitas)} | Master: {len(df_master)}")
print(f"Índice único: {df_master.index.is_unique}")
df_master.head()

### 6.3. Diagnóstico de la integración

Cobertura del matching por nivel y visitas con dirección que quedaron sin emparejar (candidatas a revisión manual o a ajustar `vector_tol` / `fuzzy_threshold`).

In [ ]:
matched = match_table["sitio_id"].notna()

print(f"Visitas emparejadas:    {matched.sum()} / {len(match_table)}")
print(f"Sitios EDAN con visita: {match_table.loc[matched, 'sitio_id'].nunique()} / {df_edan['sitio_id'].nunique()}")
print()
print(match_table.loc[matched, "match_method"].value_counts().to_string())

# Unmatched visitas that DO have an address — candidates for manual review
# or for loosening vector_tol / fuzzy_threshold
sin_match = df_visitas.merge(match_table.loc[~matched, ["visita_id"]], on="visita_id")
sin_match = sin_match[sin_match["direccion_norm"].astype(str).str.strip().ne("")]
print(f"\nVisitas con dirección pero sin match: {len(sin_match)}")
sin_match[["visita_id", "direccion_norm"]].head(20)